In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    cross_val_score
)

from sklearn.linear_model import (
    Ridge
)

from sklearn.ensemble import (
    StackingRegressor
)

from lightgbm import LGBMRegressor

from xgboost import XGBRegressor

In [2]:
train_df = pd.read_csv(
    "../data/processed/train_after_feature_engineering.csv"
)

In [3]:
X = train_df.drop("SalePrice", axis=1)

y = np.log1p(train_df["SalePrice"])

In [4]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [5]:
def rmse_cv(model):

    rmse = np.sqrt(
        -cross_val_score(
            model,
            X,
            y,
            scoring="neg_mean_squared_error",
            cv=kf
        )
    )

    return rmse

In [6]:
ridge_model = Ridge(alpha=10)

lgbm_model = LGBMRegressor(
    learning_rate=0.01,
    n_estimators=3000,
    num_leaves=20,
    random_state=42
)

xgb_model = XGBRegressor(
    learning_rate=0.01,
    n_estimators=3000,
    max_depth=3,
    random_state=42
)

In [7]:
stack_model = StackingRegressor(

    estimators=[

        ("ridge", ridge_model),

        ("lgbm", lgbm_model),

        ("xgb", xgb_model)
    ],

    final_estimator=Ridge(alpha=10),

    cv=5,

    n_jobs=-1
)

In [8]:
stack_scores = rmse_cv(stack_model)

print("Stacking Model")

print("Mean RMSE:", stack_scores.mean())

print("Std RMSE:", stack_scores.std())

Stacking Model
Mean RMSE: 0.1406059819444649
Std RMSE: 0.022500729683396406


In [9]:
stack_model.fit(X, y)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('ridge', ...), ('lgbm', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge(alpha=10)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': no

In [10]:
import joblib

joblib.dump(
    stack_model,
    "../models/final_stacking_model.pkl"
)

['../models/final_stacking_model.pkl']

In [11]:
stacking_result = pd.DataFrame({

    "Model": ["Stacking"],

    "CV RMSE": [stack_scores.mean()]
})

stacking_result.to_csv(
    "../model_reports/stacking_results.csv",
    index=False
)

stacking_result

,Model,CV RMSE
0,Stacking,0.140606
